# mcpsafetywarden: Python API quickstart

This notebook shows how to use mcpsafetywarden from a Python script or notebook,
without the CLI or an MCP client. All 17 server tools are plain async Python functions
that you can import and call directly.

**What we cover:**
1. Setup and imports
2. Register a server
3. Inspect tools and profiles
4. Preflight risk check
5. Safe tool call (with the block/approve flow)
6. Security scan
7. Policy management
8. Run history and behavioral profile
9. Replay test

**Prerequisite:** `pip install mcpsafetywarden` (or `pip install -e .` from the repo root).
The examples use the official MCP filesystem server as the wrapped target:
`npx -y @modelcontextprotocol/server-filesystem /tmp`
Install Node.js if you don't have it, or swap in any other stdio MCP server.

## 1. Setup

In [ ]:
import asyncio
import json
import os

# Set your LLM key before importing. The wrapper reads it at call time.
# Without a key the wrapper still works but uses rule-based classification only.
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."   # or OPENAI_API_KEY / GEMINI_API_KEY

from mcpsafetywarden.server import (
    register_server,
    inspect_server,
    list_servers,
    list_server_tools,
    preflight_tool_call,
    safe_tool_call,
    get_tool_profile,
    get_retry_policy,
    suggest_safer_alternative,
    security_scan_server,
    get_security_scan,
    set_tool_policy,
    get_run_history,
    run_replay_test,
    ping_server,
)

# Helper: parse the JSON string every tool returns
def show(result: str):
    parsed = json.loads(result)
    print(json.dumps(parsed, indent=2))
    return parsed

## 2. Register a server

`register_server` connects to the wrapped server, discovers its tools, classifies each one,
and stores everything in the local SQLite database. `auto_inspect=True` (the default) does
the tool discovery step automatically.

In [ ]:
result = await register_server(
    server_id="fs",
    transport="stdio",
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem", "/tmp"],
    auto_inspect=True,
    classify_provider="anthropic",   # omit to use rule-based classification only
)
show(result)

In [ ]:
# List all registered servers
show(list_servers())

## 3. Inspect tools and profiles

`list_server_tools` returns every tool on a server with its effect class, risk level,
retry safety, and observation count.

In [ ]:
data = show(list_server_tools("fs"))

# Quick summary table
print(f"\n{'Tool':<30} {'Effect':<20} {'Risk':<12} {'Runs'}")
print("-" * 70)
for t in data.get("tools", []):
    print(f"{t['tool_name']:<30} {t.get('effect_class','?'):<20} {t.get('risk_level','?'):<12} {t.get('run_count', 0)}")

In [ ]:
# Full profile for a single tool
show(get_tool_profile("fs", "write_file"))

## 4. Preflight risk check

Call `preflight_tool_call` before executing anything sensitive. It returns the risk level,
whether approval is recommended, expected latency band, and any security scan findings
for that specific tool.

In [ ]:
preflight = show(await preflight_tool_call(server_id="fs", tool_name="write_file"))

assessment = preflight["assessment"]
print(f"\nRisk level         : {assessment['risk_level']}")
print(f"Approval needed    : {assessment['approval_recommended']}")
print(f"Effect class       : {assessment['likely_effect']}")
print(f"Destructiveness    : {assessment['likely_destructiveness']}")
print(f"Expected latency   : {assessment['expected_latency_band']}")
print(f"Data source        : {preflight['data_source']}")

In [ ]:
# Low-risk tool, no approval needed
show(await preflight_tool_call(server_id="fs", tool_name="read_file"))

## 5. Safe tool call

`safe_tool_call` is the main execution path. It runs preflight, scans arguments for 9
injection categories (38 patterns), forwards the call, scans output, and records telemetry.

For low/medium-low risk tools it executes immediately. For medium/high risk tools it
returns a blocked response with an alternatives list. Re-call with `approved=True` to
proceed anyway.

In [ ]:
# Low-risk tool, executes immediately
result = show(await safe_tool_call(
    server_id="fs",
    tool_name="read_file",
    args={"path": "/tmp/hello.txt"},
))

print("\nExecution context:", result.get("execution_context"))
print("Latency ms:       ", result.get("latency_ms"))

In [ ]:
# Higher-risk tool: first call returns a block with alternatives
blocked = show(await safe_tool_call(
    server_id="fs",
    tool_name="write_file",
    args={"path": "/tmp/out.txt", "content": "hello"},
    llm_provider="anthropic",
))

if blocked.get("blocked"):
    print("\nBlocked. Alternatives:")
    for i, alt in enumerate(blocked.get("alternatives", []), 1):
        print(f"  {i}. {alt['tool_name']}  risk_reduction={alt['risk_reduction']}  coverage={alt['coverage']}")

In [ ]:
# Option A: use a suggested alternative
if blocked.get("blocked") and blocked.get("alternatives"):
    alt_name = blocked["alternatives"][0]["tool_name"]
    show(await safe_tool_call(
        server_id="fs",
        tool_name="write_file",
        args={"path": "/tmp/out.txt", "content": "hello"},
        use_alternative=alt_name,
    ))

In [ ]:
# Option B: approve and proceed despite the risk level
show(await safe_tool_call(
    server_id="fs",
    tool_name="write_file",
    args={"path": "/tmp/out.txt", "content": "hello"},
    approved=True,
))

## 6. Security scan

Run the mcpsafety+ five-stage pipeline (Recon, Planner, Hacker, Auditor, Supervisor)
against the registered server. This sends real payloads to the live server.
`confirm_authorized=True` is required: it is your acknowledgement that you have
permission to probe this server.

In [ ]:
scan = show(await security_scan_server(
    server_id="fs",
    provider="anthropic",
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
    confirm_authorized=True,
    skip_web_research=True,   # set False to enrich with CVE/Arxiv lookups
))

print(f"\nOverall risk : {scan.get('overall_risk_level')}")
print(f"Summary      : {scan.get('summary')}")
print(f"Findings     : {len(scan.get('tool_findings', []))}")

In [ ]:
# Print each finding
for f in scan.get("tool_findings", []):
    print(f"[{f['risk_level']}] {f.get('finding')}")
    print(f"  Exploitation : {f.get('exploitation_scenario')}")
    print(f"  Remediation  : {f.get('remediation')}")
    print()

In [ ]:
# Retrieve the stored scan report later (no re-scan needed)
show(get_security_scan("fs"))

## 7. Policy management

Set permanent allow or block policies on individual tools. A `block` policy rejects
the tool unconditionally on every future call. An `allow` policy bypasses the risk gate
(argument scanning still runs).

In [ ]:
# Block a tool permanently
show(set_tool_policy(server_id="fs", tool_name="delete_file", policy="block"))

In [ ]:
# This will now be rejected immediately without running preflight
show(await safe_tool_call(
    server_id="fs",
    tool_name="delete_file",
    args={"path": "/tmp/out.txt"},
))

In [ ]:
# Allow a low-risk tool to skip the risk gate entirely
show(set_tool_policy(server_id="fs", tool_name="read_file", policy="allow"))

# Remove the policy and return to normal flow
show(set_tool_policy(server_id="fs", tool_name="read_file", policy="clear"))

## 8. Run history and behavioral profile

Every proxied call updates the tool's behavioral profile: p50/p95 latency, failure rate,
output size, schema stability. The profile feeds back into preflight risk assessment and
retry policy recommendations.

In [ ]:
# Recent run history
history = show(get_run_history(server_id="fs", tool_name="read_file", limit=10))

runs = history.get("runs", [])
if runs:
    print(f"\n{'Run ID':<38} {'OK':<6} {'Latency ms':<14} {'Output bytes'}")
    print("-" * 75)
    for r in runs:
        print(f"{r['run_id']:<38} {str(r['success']):<6} {str(r.get('latency_ms','')):<14} {r.get('output_size','')}")

In [ ]:
# Full behavioral profile: shows observed stats once enough calls have been proxied
profile = show(get_tool_profile("fs", "read_file"))

stats = profile.get("observed_stats") or {}
if stats:
    print(f"\nRun count      : {stats.get('run_count')}")
    print(f"Failure rate   : {stats.get('failure_rate'):.1%}" if stats.get('failure_rate') is not None else "Failure rate   : n/a")
    print(f"Latency p50    : {stats.get('latency_p50_ms')} ms")
    print(f"Latency p95    : {stats.get('latency_p95_ms')} ms")
    print(f"Output p95     : {stats.get('output_size_p95_bytes')} bytes")
    print(f"Schema stable  : {stats.get('schema_stability'):.0%}" if stats.get('schema_stability') is not None else "Schema stable  : n/a")

In [ ]:
# Retry policy recommendation
show(await get_retry_policy(
    server_id="fs",
    tool_name="read_file",
    llm_provider="anthropic",
))

## 9. Replay test

`run_replay_test` calls the tool twice with a 0.5 s delay and compares the outputs.
Useful for verifying idempotency before letting an agent retry on failure.
`confirm_authorized=True` is required.

In [ ]:
replay = show(await run_replay_test(
    server_id="fs",
    tool_name="read_file",
    args={"path": "/tmp/hello.txt"},
    confirm_authorized=True,
))

print(f"\nVerdict: {replay.get('verdict')}")

## 10. Ping server

Check reachability. For SSE/streamable_http servers with a Kali MCP server registered,
this also runs `quick_scan` and `traceroute` against the target host.

In [ ]:
show(await ping_server("fs"))

## Scripting pattern

If you are not in a notebook, wrap everything in `asyncio.run()`:

In [ ]:
import asyncio, json, os
from mcpsafetywarden.server import register_server, safe_tool_call

async def main():
    await register_server(
        server_id="fs",
        transport="stdio",
        command="npx",
        args=["-y", "@modelcontextprotocol/server-filesystem", "/tmp"],
    )
    result = json.loads(await safe_tool_call(
        server_id="fs",
        tool_name="read_file",
        args={"path": "/tmp/hello.txt"},
    ))
    print(result)

# asyncio.run(main())
# Commented out: would run the full flow. Uncomment in a .py script.